# CEFR Classification — Stream B: DeBERTa + LLM (Kaggle P100)

**Complement to `kaggle_run.ipynb` (Stream A: Exp 0–4, 7–9)**  
This notebook runs Stream B experiments on a **Kaggle P100 (16 GB, fp16)**.

| # | Experiment | Type | GPU | Est. time/lang |
|---|---|---|---|---|
| Exp 5 | Hybrid Essay Aggregation | CPU | — | ~2 min |
| Exp 6 | Domain Transfer (TF-IDF+LR) | CPU | — | ~3 min |
| Exp 10 | Ensemble LR + ComplementNB | CPU | — | ~2 min |
| Exp 11 | DeBERTa-v3-base CE fine-tune | GPU | P100 fp16 | ~25 min |
| Exp 12 | DeBERTa-v3-base + CORAL ordinal | GPU | P100 fp16 | ~25 min |
| Exp 13 | XLM-R + Ordinal Late Fusion | GPU | P100 fp16 | ~45 min |
| Exp 14 | LLaMA-3.2-3B LoRA ×2 seeds | GPU | P100 fp16 | ~90 min |

> **Total (1 language):** ~3.2 h — P100 session ~9 h → fits `en` + one more language.  
> **HF_TOKEN** — add as Kaggle Secret (needed only for Exp 14 / LLaMA).  
> Accept [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) license on HF Hub before running Exp 14.

## 1. Environment Setup

In [ ]:
import subprocess, sys

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", result.stdout.strip() or "No GPU detected")

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
    print(f"Device  : {name} ({vram:.1f} GB)")
    print(f"bf16    : {'supported' if bf16_ok else 'NOT supported — will use fp16'}")
    if 'P100' not in name and 'Tesla' not in name and not bf16_ok:
        print("⚠️  Expected P100; bf16 not available → fp16 mode forced")
else:
    print("No GPU — GPU experiments will be skipped")

In [ ]:
# bitsandbytes is intentionally excluded here — its CUDA kernels do not support
# P100 (sm_60, compute capability 6.0) in versions >=0.39, which causes a
# "no kernel image for device" error that corrupts the entire CUDA context.
# It is installed lazily only in the Exp 14 cell (LLaMA QLoRA).
!pip install -q \
    "datasets>=2.14.0" \
    "transformers>=4.44.0" \
    "accelerate>=0.30.0" \
    "peft>=0.11.0" \
    "evaluate>=0.4.0" \
    "scikit-learn>=1.3.0" \
    "langdetect>=1.0.9"

print("\n✓ Dependencies installed (bitsandbytes deferred to Exp 14)")

In [ ]:
import os, sys

REPO_URL = "https://github.com/huynhduc0/itmo-vkr-cefr.git"  # update if fork differs
REPO_DIR = "/kaggle/working/itmo-vkr-cefr"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
    print(f"Repo already at {REPO_DIR}, updated")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    import huggingface_hub
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ HF_TOKEN loaded — Exp 14 (LLaMA) available")
except Exception as e:
    print(f"⚠️  HF_TOKEN not found ({e})")
    print("   Exp 14 will be skipped. Add HF_TOKEN as a Kaggle Secret to enable it.")

## 2. Configuration

Edit the cells below to select language, task track, and which experiment groups to run.

In [ ]:
# ── User-configurable ─────────────────────────────────────────────────────────

LANGUAGE = "en"        # en | ru | it | es | de | fr
TASK     = "sentence"  # sentence | essay

# Select experiment groups to run
RUN_CPU         = True   # Exp 5, 6, 10  (fast, no GPU needed)
RUN_EXP11       = True   # DeBERTa-v3-base CE fine-tune
RUN_EXP12       = True   # DeBERTa-v3-base + CORAL ordinal
RUN_EXP13       = True   # XLM-R CE + CORAL late fusion
RUN_EXP14       = True   # LLaMA-3.2-3B LoRA × 2 seeds (requires HF_TOKEN)

# P100 overrides — fp16 only (P100 does NOT support bf16)
USE_FP16               = True   # always True on P100
BATCH_SIZE_DEBERTA     = 32     # DeBERTa-v3-base fits at bs=32 in 16 GB fp16
BATCH_SIZE_LLM         = 4      # LLaMA-3.2-3B 4-bit, per-device
GRAD_ACCUM_LLM         = 4      # effective batch = 16
NUM_EPOCHS_TRANSFORMER = 5
NUM_EPOCHS_LLM         = 2

# Domain-transfer datasets for Exp 6 (cross-corpus)
TRAIN_DATASET_6 = "UniversalCEFR/cefr_sp_en"   # always train on English
from src.config import DEFAULT_LANGUAGE_DATASETS
EVAL_DATASET_6  = DEFAULT_LANGUAGE_DATASETS.get(LANGUAGE, "UniversalCEFR/cefr_sp_en")

# ─────────────────────────────────────────────────────────────────────────────
print(f"Language : {LANGUAGE}")
print(f"Task     : {TASK}")
print(f"Exp 14   : {'enabled' if RUN_EXP14 and HF_TOKEN else 'SKIPPED (no HF_TOKEN)'}")
print(f"Exp 6 transfer: {TRAIN_DATASET_6} → {EVAL_DATASET_6}")

In [ ]:
import os
from src import config as cfg

cfg.TRANSFORMER_CONFIG["batch_size"] = BATCH_SIZE_DEBERTA
cfg.TRANSFORMER_CONFIG["num_epochs"] = NUM_EPOCHS_TRANSFORMER
cfg.LLM_CONFIG["batch_size"]          = BATCH_SIZE_LLM
cfg.LLM_CONFIG["num_epochs"]          = NUM_EPOCHS_LLM

# Do NOT set ACCELERATE_MIXED_PRECISION here.
# transformer_classifier.py auto-detects fp16 (T4/P100) vs bf16 (A100/H100)
# from the GPU's compute capability and passes it directly to TrainingArguments.
# Setting the env var separately causes Accelerate to cast model params to fp16
# before training, which breaks GradScaler ("Attempting to unscale FP16 gradients").
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import torch
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    mp = "bf16" if cc[0] >= 8 else "fp16"
    print(f"GPU    : {torch.cuda.get_device_name(0)}  (sm_{cc[0]}{cc[1]})")
    print(f"Mode   : {mp}  (auto-detected, handled inside TrainingArguments)")
else:
    print("GPU    : not available")
print(f"DeBERTa batch : {cfg.TRANSFORMER_CONFIG['batch_size']}, epochs={cfg.TRANSFORMER_CONFIG['num_epochs']}")
print(f"LLM batch     : {cfg.LLM_CONFIG['batch_size']}, grad_accum={GRAD_ACCUM_LLM}, epochs={cfg.LLM_CONFIG['num_epochs']}")

## 3. Data Preparation

In [ ]:
import subprocess

DATA_DIR = "/kaggle/working/data"

# prepare_data.py always outputs BOTH tracks (sentence/ and essay/)
# under DATA_DIR — no --task flag needed here.
cmd = [
    sys.executable, "-m", "src.prepare_data",
    "--language", LANGUAGE,
    "--output",   DATA_DIR,
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Data preparation failed")

In [ ]:
from src.run_experiments import _load_splits_from_jsonl
from src.data_utils import set_seed
from collections import Counter
from src.config import ID2LABEL

set_seed(42)
(
    (train_texts, train_labels),
    (val_texts,   val_labels),
    (test_texts,  test_labels),
) = _load_splits_from_jsonl(DATA_DIR, TASK)

print(f"Train : {len(train_texts):,}")
print(f"Val   : {len(val_texts):,}")
print(f"Test  : {len(test_texts):,}")

dist = Counter(ID2LABEL[l] for l in train_labels)
print("\nLabel distribution:", dict(sorted(dist.items())))

## 4. CPU Experiments — Exp 5, 6, 10

In [ ]:
import time
from src.run_experiments import run_exp5

all_results = []

if RUN_CPU:
    print("▶ Exp 5 – Hybrid Essay Aggregation")
    t0 = time.time()
    r5 = run_exp5(
        train_texts=train_texts,
        train_labels=train_labels,
        test_texts=test_texts,
        test_labels=test_labels,
        track=TASK,
        aggregation="mean_prob",
        seed=42,
    )
    print(f"  QWK={r5.qwk:.4f}  F1={r5.macro_f1:.4f}  Acc={r5.accuracy:.4f}  ({time.time()-t0:.1f}s)")
    all_results.append(r5)
else:
    print("Exp 5 skipped")

In [ ]:
from src.run_experiments import run_exp6

if RUN_CPU:
    print(f"▶ Exp 6 – Domain Transfer: {TRAIN_DATASET_6} → {EVAL_DATASET_6}")
    if TRAIN_DATASET_6 == EVAL_DATASET_6:
        print("  ⚠️  train and eval datasets are the same language — Exp 6 becomes in-domain (not a transfer scenario).")
        print("  Skipping to avoid misleading results. Set LANGUAGE != 'en' for a genuine transfer run.")
    else:
        t0 = time.time()
        r6 = run_exp6(
            train_dataset=TRAIN_DATASET_6,
            eval_dataset=EVAL_DATASET_6,
            track=TASK,
            seed=42,
        )
        print(f"  QWK={r6.qwk:.4f}  F1={r6.macro_f1:.4f}  Acc={r6.accuracy:.4f}  ({time.time()-t0:.1f}s)")
        all_results.append(r6)
else:
    print("Exp 6 skipped")

In [ ]:
from src.run_experiments import run_exp10

if RUN_CPU:
    print("▶ Exp 10 – Ensemble (LR + ComplementNB, soft voting)")
    t0 = time.time()
    r10 = run_exp10(
        train_texts=train_texts,
        train_labels=train_labels,
        test_texts=test_texts,
        test_labels=test_labels,
        track=TASK,
    )
    print(f"  QWK={r10.qwk:.4f}  F1={r10.macro_f1:.4f}  Acc={r10.accuracy:.4f}  ({time.time()-t0:.1f}s)")
    all_results.append(r10)
else:
    print("Exp 10 skipped")

## 5. Exp 11 — DeBERTa-v3-base CE Fine-tune (GPU, fp16)

Uses `microsoft/deberta-v3-base` — stronger than XLM-RoBERTa-base on CEFR classification.  
Cross-entropy loss, 6-class softmax output.

In [ ]:
import torch
from src.run_experiments import run_exp11

if RUN_EXP11 and torch.cuda.is_available():
    print("▶ Exp 11 – DeBERTa-v3-base CE fine-tune")
    t0 = time.time()
    r11 = run_exp11(
        train_texts=train_texts,
        train_labels=train_labels,
        val_texts=val_texts,
        val_labels=val_labels,
        test_texts=test_texts,
        test_labels=test_labels,
        track=TASK,
        language=LANGUAGE,
        num_epochs=NUM_EPOCHS_TRANSFORMER,
        batch_size=BATCH_SIZE_DEBERTA,
        seed=42,
    )
    elapsed = time.time() - t0
    print(
        f"  QWK={r11.qwk:.4f}±{r11.qwk_ci:.3f}  "
        f"F1={r11.macro_f1:.4f}±{r11.macro_f1_ci:.3f}  "
        f"Acc={r11.accuracy:.4f}  "
        f"MAE={r11.mae:.4f}  "
        f"({elapsed/60:.1f} min)"
    )
    all_results.append(r11)
elif not torch.cuda.is_available():
    print("Exp 11 skipped — no GPU available")
else:
    print("Exp 11 skipped by config")

## 6. Exp 12 — DeBERTa-v3-base + CORAL Ordinal Regression (GPU, fp16)

Same backbone as Exp 11, but with CORAL (Consistent Rank Logits) loss — enforces ordinal consistency across the 6 CEFR levels.  
Replaces the failed XLM-R + CORAL (Exp 3) with a stronger backbone and fixed threshold initialisation.

In [ ]:
from src.run_experiments import run_exp12

if RUN_EXP12 and torch.cuda.is_available():
    print("▶ Exp 12 – DeBERTa-v3-base + CORAL ordinal")
    t0 = time.time()
    r12 = run_exp12(
        train_texts=train_texts,
        train_labels=train_labels,
        val_texts=val_texts,
        val_labels=val_labels,
        test_texts=test_texts,
        test_labels=test_labels,
        track=TASK,
        language=LANGUAGE,
        num_epochs=NUM_EPOCHS_TRANSFORMER,
        batch_size=BATCH_SIZE_DEBERTA,
        seed=42,
    )
    elapsed = time.time() - t0
    print(
        f"  QWK={r12.qwk:.4f}±{r12.qwk_ci:.3f}  "
        f"F1={r12.macro_f1:.4f}±{r12.macro_f1_ci:.3f}  "
        f"Acc={r12.accuracy:.4f}  "
        f"MAE={r12.mae:.4f}  "
        f"({elapsed/60:.1f} min)"
    )
    if RUN_EXP11 and 'r11' in dir():
        delta = r12.qwk - r11.qwk
        print(f"  CORAL vs CE: ΔQWK = {delta:+.4f} ({'ordinal helps' if delta > 0 else 'CE better'})")
    all_results.append(r12)
elif not torch.cuda.is_available():
    print("Exp 12 skipped — no GPU")
else:
    print("Exp 12 skipped by config")

## 7. Exp 13 — XLM-R + Ordinal Late Fusion (GPU, fp16)

Trains two independent models (XLM-R CE + XLM-R CORAL), then **fuses predictions at inference time** via midpoint tie-breaking.  
Tests whether ensemble of loss functions outperforms a single loss.

In [ ]:
from src.run_experiments import run_exp13

if RUN_EXP13 and torch.cuda.is_available():
    print("▶ Exp 13 – XLM-R CE + CORAL late fusion")
    t0 = time.time()
    r13 = run_exp13(
        train_texts=train_texts,
        train_labels=train_labels,
        val_texts=val_texts,
        val_labels=val_labels,
        test_texts=test_texts,
        test_labels=test_labels,
        track=TASK,
        language=LANGUAGE,
        num_epochs=NUM_EPOCHS_TRANSFORMER,
        batch_size=BATCH_SIZE_DEBERTA,
        seed=42,
    )
    elapsed = time.time() - t0
    print(
        f"  QWK={r13.qwk:.4f}±{r13.qwk_ci:.3f}  "
        f"F1={r13.macro_f1:.4f}±{r13.macro_f1_ci:.3f}  "
        f"Acc={r13.accuracy:.4f}  "
        f"({elapsed/60:.1f} min)"
    )
    all_results.append(r13)
elif not torch.cuda.is_available():
    print("Exp 13 skipped — no GPU")
else:
    print("Exp 13 skipped by config")

## 8. Exp 14 — LLaMA-3.2-**1B** + LoRA Self-Consistency (GPU, 3 seeds)

Fine-tunes `meta-llama/Llama-3.2-1B-Instruct` with **QLoRA (4-bit NF4)** three times (seeds 42, 43, 44).  
Final score = mean of the three **constrained-decoding** results (direct log-prob over 6 labels).  
1B vs 3B: ~3× faster training, comparable CEFR accuracy with QLoRA fine-tuning.

**Requires** `HF_TOKEN` and accepted LLaMA-3.2 license on HF Hub.

In [ ]:
from src.run_experiments import run_exp14

if RUN_EXP14 and HF_TOKEN and torch.cuda.is_available():
    import subprocess as _sp
    _sp.run(["pip", "install", "-q", "bitsandbytes>=0.43.0"], check=True)

    print(f"
{'#'*60}")
    print("  EXP14 | Model: LLaMA-3.2-3B | 2 seeds x QLoRA 4-bit")
    print(f"{'#'*60}")
    t0 = time.time()
    r14 = run_exp14(
        train_texts=train_texts,
        train_labels=train_labels,
        val_texts=val_texts,
        val_labels=val_labels,
        test_texts=test_texts,
        test_labels=test_labels,
        track=TASK,
        language=LANGUAGE,
        seed=42,
    )
    elapsed = time.time() - t0
    print(
        f"
  [EXP14 FINAL | 3B] QWK={r14.qwk:.4f}  "
        f"F1={r14.macro_f1:.4f}  "
        f"Acc={r14.accuracy:.4f}  "
        f"({elapsed/60:.1f} min) [{r14.note}]"
    )
    all_results.append(r14)
elif not HF_TOKEN:
    print("Exp 14 skipped — HF_TOKEN not found (add as Kaggle Secret)")
elif not torch.cuda.is_available():
    print("Exp 14 skipped — no GPU")
else:
    print("Exp 14 skipped by config (set RUN_EXP14 = True)")

## 9. Results

In [ ]:
from src.run_experiments import print_comparison_table

print(f"\nStream B results — language={LANGUAGE.upper()}, task={TASK}")
print_comparison_table(all_results)

In [ ]:
import json, os
from src.run_experiments import save_results_to_files

OUT_DIR = f"/kaggle/working/results_stream_b/{TASK}/{LANGUAGE}"
save_results_to_files(all_results, OUT_DIR)

with open(os.path.join(OUT_DIR, "results.json")) as f:
    records = json.load(f)

print(f"\n{'Experiment':<48} {'QWK':>10} {'CI':>8} {'F1':>8} {'Acc':>7} {'MAE':>6}")
print("-" * 92)
for rec in records:
    ci_str = f"±{rec['qwk_ci']:.3f}" if rec.get("qwk_ci") else ""
    mae_str = f"{rec['mae']:.4f}" if rec.get("mae") else "-"
    print(
        f"{rec['name']:<48} {rec['qwk']:>10.4f} {ci_str:>8} "
        f"{rec['macro_f1']:>8.4f} {rec['accuracy']:>7.4f} {mae_str:>6}"
    )

print(f"\nResults saved → {OUT_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

COLOR_MAP = {
    "CPU":        "#9e9e9e",
    "DeBERTa":    "#4caf50",
    "CORAL":      "#5b9bd5",
    "Fusion":     "#ab47bc",
    "LLM":        "#e07b39",
}

def _color(name):
    if "LLM" in name or "LoRA" in name or "LLaMA" in name:
        return COLOR_MAP["LLM"]
    if "Fusion" in name:
        return COLOR_MAP["Fusion"]
    if "CORAL" in name:
        return COLOR_MAP["CORAL"]
    if "DeBERTa" in name or "Exp 11" in name or "Exp 12" in name:
        return COLOR_MAP["DeBERTa"]
    return COLOR_MAP["CPU"]

names  = [r["name"].replace(" – ", "\n") for r in records]
qwks   = [r["qwk"] for r in records]
cis    = [r.get("qwk_ci", 0.0) for r in records]
colors = [_color(r["name"]) for r in records]

fig, ax = plt.subplots(figsize=(13, max(4, len(names) * 0.6)))
y    = np.arange(len(names))
bars = ax.barh(y, qwks, xerr=cis, align="center", height=0.6,
               color=colors, capsize=4, error_kw={"elinewidth": 1.5})

ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Quadratic Weighted Kappa (QWK)")
ax.set_title(
    f"CEFR Stream B — {LANGUAGE.upper()} / {TASK}  (QWK ± 95 % CI)",
    fontsize=12,
)
ax.set_xlim(0, 1.05)
ax.axvline(x=0, color="black", linewidth=0.8)
ax.invert_yaxis()

for bar, v, ci in zip(bars, qwks, cis):
    label = f"{v:.3f}" + (f"±{ci:.3f}" if ci else "")
    ax.text(
        min(v + 0.01, 1.03), bar.get_y() + bar.get_height() / 2,
        label, va="center", fontsize=8,
    )

legend_handles = [
    mpatches.Patch(color=COLOR_MAP["CPU"],     label="Baseline (CPU)"),
    mpatches.Patch(color=COLOR_MAP["DeBERTa"], label="DeBERTa-v3 CE"),
    mpatches.Patch(color=COLOR_MAP["CORAL"],   label="DeBERTa-v3 CORAL"),
    mpatches.Patch(color=COLOR_MAP["Fusion"],  label="Late Fusion"),
    mpatches.Patch(color=COLOR_MAP["LLM"],     label="LLaMA+LoRA"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9)
plt.tight_layout()

plot_path = os.path.join(OUT_DIR, "qwk_bar_stream_b.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved → {plot_path}")

In [ ]:
print("=" * 64)
print("Stream B session summary")
print(f"  Language : {LANGUAGE.upper()}")
print(f"  Task     : {TASK}")
print(f"  Results  : {OUT_DIR}")
if torch.cuda.is_available():
    print(f"  GPU      : {torch.cuda.get_device_name(0)}")
if all_results:
    best = max(all_results, key=lambda r: r.qwk)
    ci   = getattr(best, 'qwk_ci', 0.0) or 0.0
    print(f"  Best QWK : {best.qwk:.4f}±{ci:.3f} ({best.name})")
print("=" * 64)

## 10. (Optional) Multilingual Loop

Uncomment to sweep all 6 languages.  
Runs **Exp 10, 11, 12** per language (skips Exp 14 to stay within the 9-hour P100 limit).  
Results are saved per-language under `/kaggle/working/results_stream_b/{task}/{lang}/`.

In [ ]:
# UNCOMMENT to run multilingual sweep (Exp 10 + 11 + 12 for all 6 languages)
#
# from src.config import SUPPORTED_LANGUAGES
# from src.run_experiments import (
#     run_exp10, run_exp11, run_exp12,
#     _load_splits_from_jsonl, save_results_to_files, print_comparison_table,
# )
# from src.data_utils import set_seed
# import subprocess, time
#
# ALL_LANG_RESULTS = {}
#
# for lang in SUPPORTED_LANGUAGES:
#     print(f"\n{'='*60}\nLanguage: {lang.upper()}\n{'='*60}")
#
#     lang_data_dir = f"/kaggle/working/data_{lang}"
#     subprocess.run(
#         [sys.executable, "-m", "src.prepare_data",
#          "--language", lang, "--output", lang_data_dir],
#         cwd=REPO_DIR, capture_output=True,
#     )
#
#     (tr_t, tr_l), (vl_t, vl_l), (te_t, te_l) = _load_splits_from_jsonl(lang_data_dir, TASK)
#     set_seed(42)
#
#     lang_results = []
#     lang_results.append(run_exp10(tr_t, tr_l, te_t, te_l, track=TASK))
#
#     if torch.cuda.is_available():
#         lang_results.append(
#             run_exp11(tr_t, tr_l, vl_t, vl_l, te_t, te_l,
#                       track=TASK, language=lang,
#                       num_epochs=NUM_EPOCHS_TRANSFORMER,
#                       batch_size=BATCH_SIZE_DEBERTA, seed=42)
#         )
#         lang_results.append(
#             run_exp12(tr_t, tr_l, vl_t, vl_l, te_t, te_l,
#                       track=TASK, language=lang,
#                       num_epochs=NUM_EPOCHS_TRANSFORMER,
#                       batch_size=BATCH_SIZE_DEBERTA, seed=42)
#         )
#
#     save_results_to_files(lang_results, f"/kaggle/working/results_stream_b/{TASK}/{lang}")
#     ALL_LANG_RESULTS[lang] = lang_results
#     print_comparison_table(lang_results)
#
# print("\n✓ Multilingual sweep done")

print("Multilingual loop cell — uncomment to run")